# Exercise: Preparing Text Data for nanoGPT

Welcome! In this exercise, you'll put into practice the data preparation steps required for training a GPT model. You'll work with raw text, tokenize it, and save it in a memory-efficient binary format, just like in the nanoGPT project. This is a crucial first step in any language model training pipeline.

**In this exercise you will:**
*   Implement a function to encode raw text into a sequence of integer token IDs.
*   Implement a function to save these token IDs to a binary file using `numpy.memmap`.
*   Combine these functions to build a small data-processing pipeline.

Let's get started!

In [ ]:
import numpy as np
import os
import tempfile
from typing import List

# Helper class: A simple character-level tokenizer
# In a real-world scenario, you would use a more advanced tokenizer like tiktoken (BPE).
# For this exercise, we'll use this simple one to focus on the data pipeline.
# You do not need to modify this class.
class SimpleCharacterTokenizer:
    """A simple character-level tokenizer for demonstration."""
    def __init__(self, corpus: str):
        # Create a sorted, unique list of characters from the corpus
        self.chars = sorted(list(set(corpus)))
        self.vocab_size = len(self.chars)
        # Create mappings from character to integer and vice-versa
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for i, ch in enumerate(self.chars)}

    def encode(self, s: str) -> List[int]:
        """Encodes a string into a list of token IDs."""
        return [self.stoi[c] for c in s if c in self.stoi]

    def decode(self, ids: List[int]) -> str:
        """Decodes a list of token IDs back into a string."""
        return ''.join([self.itos[i] for i in ids])

print("Setup complete. The SimpleCharacterTokenizer is ready to use.")

### Exercise 1: Encode Text

Your first task is to create a function that takes a raw string and a tokenizer object, and returns a list of integer token IDs. This is the "encoding" or "tokenization" step.

**Instructions:**
- The function `encode_text` takes two arguments: `text` (a string) and `tokenizer` (an object with an `.encode()` method).
- You should call the tokenizer's `.encode()` method on the input text to get the list of token IDs.
- Return this list of integers.

**Hint:** This can be done in a single line of code!

In [ ]:
def encode_text(text: str, tokenizer) -> List[int]:
    """
    Encodes a string of text into a list of integer token IDs using a given tokenizer.

    Args:
        text (str): The input string to encode.
        tokenizer: An object with an .encode() method that returns a list of integers.

    Returns:
        List[int]: A list of token IDs.
    """
        tokenizer: An object with an .encode() method that returns a list of integers.

    Returns:
        List[int]: A list of token IDs.
    """
    ### START CODE HERE ###
    token_ids = tokenizer.encode(text)
    return token_ids
    ### END CODE HERE ###

In [ ]:
# Public test for encode_text
print("Running tests for encode_text...")

# 1. Create a tokenizer based on a small corpus
corpus = "hello world"
tokenizer = SimpleCharacterTokenizer(corpus)
# The vocabulary will be: ' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w'
# stoi: {' ': 0, 'd': 1, 'e': 2, 'h': 3, 'l': 4, 'o': 5, 'r': 6, 'w': 7}

# 2. Test Case 1: Simple encoding
text1 = "hello"
expected1 = [3, 2, 4, 4, 5]
result1 = encode_text(text1, tokenizer)
assert result1 == expected1, f"Test Case 1 Failed: Got {result1}, expected {expected1}"
print("✅ Test Case 1 Passed")

# 3. Test Case 2: Encoding with spaces and different characters
text2 = "world"
expected2 = [7, 5, 6, 4, 1]
result2 = encode_text(text2, tokenizer)
assert result2 == expected2, f"Test Case 2 Failed: Got {result2}, expected {expected2}"
print("✅ Test Case 2 Passed")

# 4. Test Case 3: Encoding a string with characters not in the vocabulary
# The SimpleCharacterTokenizer will ignore unknown characters.
text3 = "hello there!"
expected3 = [3, 2, 4, 4, 5, 0, 3, 2, 6, 2] # "!" and "t" are not in the corpus
result3 = encode_text(text3, tokenizer)
assert result3 == expected3, f"Test Case 3 Failed: Got {result3}, expected {expected3}"
print("✅ Test Case 3 Passed")

print("\nAll tests passed for encode_text! Great job!")

### Exercise 2: Save to Binary File

Excellent! Now that you have token IDs, the next step is to save them to a file. For large datasets, using a standard text file is inefficient. Instead, we'll save the data as a compact binary file. We'll use `numpy.memmap`, which creates a memory-map to a file. This is highly efficient as it doesn't require loading the entire dataset into RAM.

**Instructions:**
- The function `save_to_binary` takes a NumPy array of `token_ids`, an `output_path`, and the `dtype` for the array elements.
- Create a `numpy.memmap` object with the specified `output_path`, `dtype`, `mode='w+'` (write mode), and the correct `shape`.
- Assign the `token_ids` to the memory-mapped array. A simple slice `[:]` is effective for this.
- Finally, call `.flush()` on the memmap object to ensure all changes are written from memory to the file on disk.

In [ ]:
def save_to_binary(token_ids: np.ndarray, output_path: str, dtype=np.uint16) -> None:
    """
    Saves a NumPy array of token IDs to a binary file using numpy.memmap.

    Args:
        token_ids (np.ndarray): The array of token IDs to save.
        output_path (str): The path to the output binary file.
        dtype (np.dtype, optional): The NumPy data type to use for saving. Defaults to np.uint16.
    """
        output_path (str): The path to the output binary file.
        dtype (np.dtype, optional): The NumPy data type to use for saving. Defaults to np.uint16.
    """
    ### START CODE HERE ###
    # Get the total number of tokens
    n = len(token_ids)
    
    # Create a memory-mapped array in write mode ('w+')
    memmap_arr = np.memmap(output_path, dtype=dtype, mode='w+', shape=(n,))
    
    # Write the token_ids to the memory-mapped array
    memmap_arr[:] = token_ids
    
    # Flush changes to disk
    memmap_arr.flush()
    ### END CODE HERE ###

In [ ]:
# Public test for save_to_binary
print("Running tests for save_to_binary...")

# Use a temporary file for testing so we don't create files in the directory
with tempfile.NamedTemporaryFile(suffix=".bin", delete=True) as tmpfile:
    output_path = tmpfile.name
    
    # 1. Test Case 1: Basic save and load
    test_data1 = np.array([10, 20, 30, 40, 50], dtype=np.uint16)
    
    # Save the data using your function
    save_to_binary(test_data1, output_path, dtype=np.uint16)
    
    # Load the data back using np.memmap to verify
    loaded_data1 = np.memmap(output_path, dtype=np.uint16, mode='r')
    
    assert os.path.exists(output_path), "Test Case 1 Failed: Output file was not created."
    assert np.array_equal(loaded_data1, test_data1), f"Test Case 1 Failed: Loaded data {loaded_data1} does not match original {test_data1}."
    print("✅ Test Case 1 Passed")
    
    # 2. Test Case 2: Different data type (np.int32)
    test_data2 = np.array([1000, -2000, 3000, 40000, -50000], dtype=np.int32)
    save_to_binary(test_data2, output_path, dtype=np.int32)
    loaded_data2 = np.memmap(output_path, dtype=np.int32, mode='r')
    assert np.array_equal(loaded_data2, test_data2), f"Test Case 2 Failed: Loaded data {loaded_data2} does not match original {test_data2}."
    print("✅ Test Case 2 Passed")

print("\nAll tests passed for save_to_binary! You are doing great!")

### (Optional) Challenge: Implement a Tokenizer

Feeling confident? As an optional challenge, try completing the `build_vocab` and `encode` methods of the `MySimpleTokenizer` class below. This will give you a deeper understanding of how a basic character-level tokenizer works from scratch.

**Instructions:**
1.  **`build_vocab`**:
    *   Find all unique characters in the input `corpus`.
    *   Create a sorted list of these characters.
    *   Populate `self.stoi` (string-to-integer) and `self.itos` (integer-to-string) dictionaries based on the sorted list.
2.  **`encode`**:
    *   Iterate through the input string `s`.
    *   For each character, find its corresponding integer ID from `self.stoi`.
    *   Return a list of these integer IDs. Make sure to handle characters that might not be in your vocabulary (you can just ignore them).

This challenge is optional, and you can proceed to the final integration test even if you skip it.

In [ ]:
# Putting it all together: The Full Pipeline

Now, let's combine your two functions to create a mini data preparation pipeline. We will take a sample text, encode it, and save it to a binary file. This final test ensures that your components work together seamlessly.

You don't need to write any code here. Just run the cell to see your functions in action!
This simulates the exact process used in projects like nanoGPT to prepare `train.bin` and `val.bin`.

# 1. Define sample data and tokenizer
full_text = "This is a simple sentence. We will encode and save it."
tokenizer = SimpleCharacterTokenizer(full_text)

# 2. Encode the text using your first function
token_ids_list = encode_text(full_text, tokenizer)
token_ids_array = np.array(token_ids_list, dtype=np.uint16)

print(f"Original text: '{full_text}'")
print(f"Encoded token IDs: {token_ids_array}")
print("-" * 20)

# 3. Save the tokens to a binary file using your second function
with tempfile.NamedTemporaryFile(suffix=".bin", delete=True) as tmpfile:
    bin_path = tmpfile.name
    save_to_binary(token_ids_array, bin_path)
    print(f"Token IDs saved to temporary file: {bin_path}")

    # 4. Verification: Load the data and check if it's correct
    loaded_tokens = np.memmap(bin_path, dtype=np.uint16, mode='r')
    print(f"Loaded token IDs: {loaded_tokens}")
    
    assert np.array_equal(token_ids_array, loaded_tokens), "Verification Failed: Loaded tokens do not match original tokens."
    
    # 5. Decode back to text to see the result
    decoded_text = tokenizer.decode(loaded_tokens.tolist())
    print(f"Decoded text: '{decoded_text}'")
    
    # The SimpleCharacterTokenizer is lossless if all characters are in the vocab
    assert decoded_text == full_text, "Verification Failed: Decoded text does not match original text."

print("\n🎉 Congratulations! You have successfully implemented the data preparation pipeline! 🎉")